# Pipeline de entrega — ResNet50 v2 + Grad-CAM / Integrated Gradients

**Pergunta do projeto:** as regiões que o modelo destaca para classificar um nódulo pulmonar correspondem ao nódulo anotado pelos radiologistas?

Este notebook é a **entrada única** da entrega. Ele:

1. resume o tratamento de dados do pipeline v2;
2. carrega o checkpoint da **ResNet50 v2** (sem retreino);
3. gera mapas **Grad-CAM** e **Integrated Gradients** em casos do teste;
4. no final, mostra tabelas, gráficos e overlays com a ROI do radiologista.

**Pré-requisitos locais**

- `models/resnet50_v2/best_model.pth`
- `outputs/sprint5/split_manifest_v2.csv`, predições e (opcional) cache de anotações
- DICOM LIDC em `data/LIDC-IDRI/` para regenerar overlays ao vivo


## 1. Setup

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
from captum.attr import IntegratedGradients
from IPython.display import display
from scipy.ndimage import gaussian_filter
from torchvision import models

def find_project_root() -> Path:
    """Resolve PI3-Grupo-03 even if the notebook cwd is notebooks/ or the outer Biopark folder."""
    here = Path.cwd().resolve()
    candidates = [here, *here.parents]
    if here.name == "notebooks":
        candidates.insert(0, here.parent)
    for path in candidates:
        if (path / "models" / "resnet50_v2" / "config.json").exists() and (path / "scripts" / "sprint5_xai_v2.py").exists():
            return path
        nested = path / "PI3-Grupo-03"
        if (nested / "models" / "resnet50_v2" / "config.json").exists():
            return nested
    raise FileNotFoundError(
        f"Nao achei a raiz do projeto a partir de {here}. "
        "Abra o notebook com o kernel do .venv em PI3-Grupo-03."
    )

ROOT = find_project_root()
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "scripts"))

from src.xai.gradcam import GradCAM
from sprint5_crop_preview import contour_v2
from sprint5_retrain_resnet50 import CROP_MM, IMAGE_SIZE, extract_crop
from sprint5_xai_v2 import MAPS, MAP_NAMES, center_map, normalise, to_input

SEED = 42
OUT5 = ROOT / "outputs" / "sprint5"
CKPT = ROOT / "models" / "resnet50_v2" / "best_model.pth"
ANN_CACHE = OUT5 / "matched_annotations_cache.pkl"
FIG_DIR = ROOT / "docs" / "entrega" / "figures"
N_LIVE = 6

assert CKPT.exists(), f"Checkpoint ausente: {CKPT}"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("ROOT:", ROOT)
print("device:", device)
print("checkpoint:", CKPT, f"({CKPT.stat().st_size / 1e6:.1f} MB)")


ROOT: C:\Users\Administrator\Documents\Biopark\Projeto-Integrador-3\PI3-Grupo-03
device: cuda
checkpoint: C:\Users\Administrator\Documents\Biopark\Projeto-Integrador-3\PI3-Grupo-03\models\resnet50_v2\best_model.pth (94.4 MB)


## 2. Tratamento de dados (pipeline v2)

Regras usadas no experimento (sem retreinar aqui):

| Etapa | Decisão |
|------|---------|
| Split | 559 pacientes (391 / 84 / 84), por paciente, seed 42 |
| Nódulo físico | ROIs de leitores com bounding boxes sobrepostas na mesma fatia (union-find) |
| Rótulo | média de malignidade dos leitores do mesmo nódulo físico; maligno se média ≥ 3,5 |
| Ambíguos | 2,5 < média < 3,5 — fora do treino/validação |
| Crop | janela fixa de **64 mm** centrada na ROI, redimensionada para **224×224**, janela HU [-1000, 400] |

Os manifests e predições em `outputs/sprint5/` são a fonte oficial dos números.

In [ ]:
manifest = pd.read_csv(OUT5 / "split_manifest_v2.csv")
preds_val = pd.read_csv(OUT5 / "preds_val_resnet50_v2.csv")
preds_test = pd.read_csv(OUT5 / "preds_test_resnet50_v2.csv")
config = json.loads((ROOT / "models" / "resnet50_v2" / "config.json").read_text(encoding="utf-8"))

summary_split = (
    manifest.groupby("split")
    .agg(
        patches=("sop_uid", "size"),
        pacientes=("patient_id", "nunique"),
        malignos_v2=("label", "sum"),
        ambiguos=("is_ambiguous", "sum"),
    )
    .reindex(["train", "validation", "test"])
)
display(summary_split)
print("Config v2 (treinamento já feito):")
for k in ("split", "label", "crop", "best_val_auc_nodule", "best_epoch", "seed"):
    print(f"  {k}: {config.get(k)}")


,patches,pacientes,malignos_v2,ambiguos
split,,,,
train,18508,391,7203,6615
validation,3694,84,1773,1056
test,4163,84,1723,1487


Config v2 (treinamento já feito):
  split: 559 patients (391/84/84), seed 42 (same as notebook)
  label: mean malignancy of readers on the same physical nodule >= 3.5
  crop: 64.0 mm square around ROI centre, resized to 224, window [-1000.0, 400.0] HU
  best_val_auc_nodule: 0.9220236765691312
  best_epoch: 1
  seed: 42


## 3. Carregar anotações e montar o conjunto de avaliação

Usa o cache `matched_annotations_cache.pkl` (pontos da ROI + caminho DICOM).
Se o cache não existir, reconstrói a partir do notebook legado do split (demorado).

In [ ]:
def load_annotations() -> pd.DataFrame:
    if ANN_CACHE.exists():
        print("Cache de anotações:", ANN_CACHE)
        return pd.read_pickle(ANN_CACHE)
    from sprint5_predictions import run_notebook_until_split

    print("Cache ausente — executando notebook até o split (pode demorar)...")
    ns = run_notebook_until_split()
    ann = ns["matched_annotations"][["sop_uid", "dicom_path", "points"]].copy()
    ANN_CACHE.parent.mkdir(parents=True, exist_ok=True)
    ann.to_pickle(ANN_CACHE)
    print("Cache gravado em", ANN_CACHE)
    return ann


annotations = load_annotations()


def attach_split(name: str, preds: pd.DataFrame) -> pd.DataFrame:
    man = manifest[manifest["split"] == name].reset_index(drop=True)
    frame = man.copy()
    frame["prob_malignant"] = preds["prob_malignant"].to_numpy()
    ann = annotations.loc[frame["annotation_index"]]
    if not (ann["sop_uid"].astype(str).to_numpy() == frame["sop_uid"].astype(str).to_numpy()).all():
        raise SystemExit(f"{name}: annotation_index desalinhado")
    frame["dicom_path"] = ann["dicom_path"].to_numpy()
    frame["points"] = pd.Series(
        [np.asarray(p, dtype=np.float32) for p in ann["points"]],
        index=frame.index,
        dtype=object,
    )
    return frame


val_df = attach_split("validation", preds_val)
test_df = attach_split("test", preds_test)
print("validation:", len(val_df), "| test:", len(test_df))
print("exemplo dicom existe?", Path(str(test_df.iloc[0]["dicom_path"])).exists())


Cache de anotações: C:\Users\Administrator\Documents\Biopark\Projeto-Integrador-3\PI3-Grupo-03\outputs\sprint5\matched_annotations_cache.pkl
validation: 3694 | test: 4163
exemplo dicom existe? True


## 4. Modelo ResNet50 v2 (somente inferência)

In [ ]:
model = models.resnet50(weights=None)
model.fc = nn.Linear(model.fc.in_features, 2)
model.load_state_dict(torch.load(CKPT, map_location=device))
model = model.to(device).eval()
print("ResNet50 v2 carregada.")


ResNet50 v2 carregada.


## 5. Explicabilidade ao vivo (amostra do teste)

Para cada nódulo físico, o experimento completo usa o maior ROI do leitor.
Aqui regeneramos Grad-CAM (`layer4`) e Integrated Gradients (baseline de ar, 32 passos)
em uma amostra, e comparamos com o mapa **centrado** (baseline sem aprendizado).

Métricas do conjunto completo: `outputs/sprint5/xai_v2_*.csv`.

In [ ]:
def case_maps(crop, pred, gradcam, ig, baseline, center, rng):
    x = to_input(crop).to(device)
    cam = gradcam.generate(x, target_index=pred).cam
    attr = ig.attribute(
        x.clone().requires_grad_(True),
        baselines=baseline,
        target=pred,
        n_steps=32,
        internal_batch_size=8,
    )
    ig_map = normalise(attr.abs().sum(dim=1)[0].detach().cpu().numpy())
    return {
        "gradcam": normalise(cam),
        "ig": ig_map,
        "random": normalise(gaussian_filter(rng.random((IMAGE_SIZE, IMAGE_SIZE)), sigma=8)),
        "center": center,
    }


def build_mask(points, cx, cy, spacing):
    mask = np.zeros((IMAGE_SIZE, IMAGE_SIZE), np.uint8)
    cv2.fillPoly(mask, [np.round(contour_v2(points, cx, cy, spacing)).astype(np.int32)], 1)
    return mask


per_case = pd.read_csv(OUT5 / "xai_v2_per_case.csv")
test_metrics = per_case[per_case["split"] == "test"].copy()
for c in ("patient_id", "sop_uid", "reading_session", "nodule_id"):
    test_metrics[c] = test_metrics[c].astype(str)
    test_df[c] = test_df[c].astype(str)

t_cam = 0.5
chosen_idx = (
    list(test_metrics.sort_values(f"gradcam_iou@{t_cam}").index[-2:])
    + list(
        test_metrics[test_metrics["gradcam_pg_hit"] == 0]
        .sample(min(2, int((test_metrics["gradcam_pg_hit"] == 0).sum())), random_state=SEED)
        .index
    )
    + list(
        test_metrics.assign(
            acerto=lambda d: np.where(
                d["is_ambiguous"] == 1,
                "amb",
                np.where(d["pred"] == d["label"], "ok", "erro"),
            )
        )
        .query("acerto == 'erro'")
        .sample(min(2, N_LIVE), random_state=SEED)
        .index
    )
)
chosen_idx = list(dict.fromkeys(chosen_idx))[:N_LIVE]
chosen_rows = test_metrics.loc[chosen_idx]
print("Casos escolhidos para overlay ao vivo:")
display(
    chosen_rows[
        [
            "patient_id",
            "label",
            "pred",
            "prob_malignant",
            "size_mm",
            f"gradcam_iou@{t_cam}",
            "gradcam_pg_hit",
            "ig_iou@0.1",
            "ig_pg_hit",
        ]
    ]
)


Casos escolhidos para overlay ao vivo:


,patient_id,label,pred,prob_malignant,size_mm,gradcam_iou@0.5,gradcam_pg_hit,ig_iou@0.1,ig_pg_hit
264,LIDC-IDRI-0138,0,1,0.753238,23.730463,0.554226,1,0.304340,1
271,LIDC-IDRI-0583,1,1,0.770473,12.773438,0.564258,1,0.285045,1
401,LIDC-IDRI-0662,0,0,0.014627,7.734375,0.001244,0,0.039174,0
509,LIDC-IDRI-0335,0,0,0.010206,3.574219,0.020637,0,0.028518,0
355,LIDC-IDRI-0725,1,0,0.433852,9.140625,0.000000,0,0.090143,0
325,LIDC-IDRI-0055,1,0,0.267662,9.843750,0.000000,0,0.210695,1


In [ ]:
key = ["patient_id", "sop_uid", "reading_session", "nodule_id"]
live = chosen_rows.merge(test_df[key + ["dicom_path", "points"]], on=key, how="left")
print("casos com dicom_path:", live["dicom_path"].notna().sum(), "/", len(live))

gradcam = GradCAM(model, model.layer4, apply_sigmoid=False)
ig = IntegratedGradients(model)
baseline = to_input(np.zeros((IMAGE_SIZE, IMAGE_SIZE), np.float32)).to(device)
center = center_map()
rng = np.random.default_rng(SEED)

examples = []
for _, row in live.iterrows():
    path = row.get("dicom_path")
    if not isinstance(path, str) or not Path(path).exists():
        print("DICOM ausente:", row.get("patient_id"), row.get("sop_uid"))
        continue
    points = np.asarray(row["points"], dtype=np.float32)
    cx, cy = float(points[:, 0].mean()), float(points[:, 1].mean())
    spacing = tuple(float(v) for v in pydicom.dcmread(path, stop_before_pixels=True).PixelSpacing)
    crop = extract_crop(path, cx, cy).astype(np.float32)
    mask = build_mask(points, cx, cy, spacing)
    if mask.sum() == 0:
        continue
    x = to_input(crop).to(device)
    with torch.no_grad():
        prob = float(torch.softmax(model(x), dim=1)[0, 1].item())
    pred = int(prob >= 0.5)
    maps = case_maps(crop, pred, gradcam, ig, baseline, center, rng)
    examples.append({"row": row, "crop": crop, "mask": mask, "maps": maps, "prob": prob, "pred": pred})
    print(f"OK {row['patient_id']} p={prob:.3f} pred={pred} label={int(row['label'])}")

gradcam.close()
print("Overlays gerados:", len(examples))


casos com dicom_path: 6 / 6
OK LIDC-IDRI-0138 p=0.753 pred=1 label=0
OK LIDC-IDRI-0583 p=0.770 pred=1 label=1
OK LIDC-IDRI-0662 p=0.015 pred=0 label=0
OK LIDC-IDRI-0335 p=0.010 pred=0 label=0
OK LIDC-IDRI-0725 p=0.434 pred=0 label=1
OK LIDC-IDRI-0055 p=0.268 pred=0 label=1
Overlays gerados: 6


## 6. Resultados finais — tabelas, gráficos e imagens

### 6.1 Métricas agregadas (teste completo)

Limiares de IoU escolhidos na validação: Grad-CAM 0,5 · IG 0,1 · aleatório 0,2 · centrado 0,9.

O mapa **centrado** vence em IoU/Pointing Game porque o crop é centrado no nódulo —
por isso ele é a referência, não um alvo absoluto de IoU ≥ 0,40.

In [ ]:
summary = pd.read_csv(OUT5 / "xai_v2_summary.csv")
sweep = pd.read_csv(OUT5 / "xai_v2_threshold_sweep.csv")
display(
    summary[summary["grupo"] == "teste (todos)"][
        [
            "mapa",
            "limiar",
            "n_casos",
            "IoU",
            "IoU IC",
            "Dice",
            "Pointing Game",
            "Pico dentro da máscara",
            "Distância do pico (mm)",
        ]
    ].round(3)
)

metrics_model = pd.read_csv(OUT5 / "metrics_provisorio_v2_todos.csv")
print("\nClassificação ResNet50 v2 (provisório) — referência:")
display(metrics_model.head(12))


,mapa,limiar,n_casos,IoU,IoU IC,Dice,Pointing Game,Pico dentro da máscara,Distância do pico (mm)
0,Grad-CAM,0.5,275,0.112,[0.093; 0.133],0.180,0.422,0.302,9.016
1,Integrated Gradients,0.1,275,0.162,[0.136; 0.192],0.254,0.665,0.556,7.816
2,Aleatório,0.2,275,0.015,[0.012; 0.019],0.030,0.015,0.000,33.830
3,Centrado,0.9,275,0.501,[0.469; 0.535],0.642,1.000,0.996,0.546



Classificação ResNet50 v2 (provisório) — referência:


,status,unit,model,threshold,metric,role,value,ci_lower,ci_upper,n_boot_valid
0,provisorio,nodule (patient|study|reader|nodule_id),resnet50_v2,0.5,auc,primary,0.9057537500549862,0.852174,0.948481,1000.0
1,provisorio,nodule (patient|study|reader|nodule_id),resnet50_v2,0.5,sensitivity,primary,0.8603351955307262,0.773998,0.931818,1000.0
2,provisorio,nodule (patient|study|reader|nodule_id),resnet50_v2,0.5,specificity,primary,0.7637795275590551,0.676226,0.853612,1000.0
3,provisorio,nodule (patient|study|reader|nodule_id),resnet50_v2,0.5,balanced_accuracy,primary,0.8120573615448907,0.746942,0.866006,1000.0
4,provisorio,nodule (patient|study|reader|nodule_id),resnet50_v2,0.5,accuracy,secondary,0.7889374090247453,0.719556,0.853165,1000.0
5,provisorio,nodule (patient|study|reader|nodule_id),resnet50_v2,0.5,precision,secondary,0.5620437956204379,0.446563,0.694490,1000.0
6,provisorio,nodule (patient|study|reader|nodule_id),resnet50_v2,0.5,f1,secondary,0.6799116997792495,0.577941,0.777784,1000.0
7,provisorio,nodule,resnet50_v2,0.5,confusion_tn_fp_fn_tp,count,388/120/25/154,NaN,NaN,NaN
8,provisorio,nodule (patient|study|reader|nodule_id),resnet50_v1,0.5,auc,primary,0.8182927902168653,0.755348,0.877454,1000.0
9,provisorio,nodule (patient|study|reader|nodule_id),resnet50_v1,0.5,sensitivity,primary,0.9776536312849162,0.947368,1.000000,1000.0


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for name in MAPS:
    for split, style in (("validation", "--"), ("test", "-")):
        s = sweep[(sweep["split"] == split) & (sweep["mapa"] == MAP_NAMES[name])]
        split_label = "val" if split == "validation" else "teste"
        ax[0].plot(s["limiar"], s["IoU"], style, label=f"{MAP_NAMES[name]} ({split_label})")
        ax[1].plot(s["limiar"], s["Dice"], style)
ax[0].set_title("IoU médio por limiar")
ax[1].set_title("Dice médio por limiar")
for a in ax:
    a.set_xlabel("limiar no mapa normalizado")
    a.grid(alpha=0.3)
ax[0].legend(fontsize=7)
fig.tight_layout()
plt.show()


C:\Users\Administrator\AppData\Local\Temp\ipykernel_10524\1766042699.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
THRESH = {"gradcam": 0.5, "ig": 0.1, "random": 0.2, "center": 0.9}

if examples:
    n_rows = len(examples)
    fig, axes = plt.subplots(
        n_rows,
        5,
        figsize=(14, 3.2 * n_rows),
        gridspec_kw={"hspace": 0.55, "wspace": 0.08},
    )
    if n_rows == 1:
        axes = np.asarray([axes])
    for r, ex in enumerate(examples):
        row, crop, mask, maps = ex["row"], ex["crop"], ex["mask"], ex["maps"]
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        panels = [("TC + máscara", None)] + [(MAP_NAMES[n], n) for n in MAPS]
        for c, (label, key) in enumerate(panels):
            a = axes[r, c]
            a.imshow(crop, cmap="gray", vmin=0, vmax=1)
            if key is not None:
                a.imshow(maps[key], cmap="jet", alpha=0.45, vmin=0, vmax=1)
                py, px = np.unravel_index(np.argmax(maps[key]), maps[key].shape)
                a.plot(px, py, "w+", ms=10, mew=2)
                t_map = THRESH[key]
                col = f"{key}_iou@{t_map}"
                iou = float(row[col]) if col in row.index else float("nan")
                pg = row.get(f"{key}_pg_hit", float("nan"))
                pg_txt = "sim" if pg == 1 else ("não" if pg == 0 else "?")
                label = f"{label}\nIoU={iou:.2f} PG={pg_txt}"
            for cnt in contours:
                a.plot(cnt[:, 0, 0], cnt[:, 0, 1], color="lime", lw=1)
            a.set_title(label, fontsize=8, pad=8)
            a.set_xticks([])
            a.set_yticks([])
        if int(row["is_ambiguous"]):
            verdict = "ambíguo"
        else:
            verdict = "maligno" if int(row["label"]) else "benigno"
        axes[r, 0].set_ylabel(
            f"{row['patient_id']}\n{verdict}, p={ex['prob']:.2f}\n~{float(row['size_mm']):.0f} mm",
            fontsize=8,
        )
    fig.suptitle(
        "ResNet50 v2: Grad-CAM / IG / baselines no teste — verde = ROI do leitor; + = pico do mapa",
        fontsize=11,
        y=1.0,
    )
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    fig.subplots_adjust(top=0.93, hspace=0.65, wspace=0.1)
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    out_png = FIG_DIR / "pipeline_entrega_exemplos.png"
    fig.savefig(out_png, dpi=100, bbox_inches="tight", pad_inches=0.25)
    plt.show()
    print("Figura salva em", out_png)
else:
    saved = FIG_DIR / "xai_v2_exemplos.png"
    print("Sem overlays ao vivo — exibindo figura salvada da Sprint 5:", saved)
    if saved.exists():
        img = plt.imread(saved)
        plt.figure(figsize=(14, 10))
        plt.imshow(img)
        plt.axis("off")
        plt.show()


C:\Users\Administrator\AppData\Local\Temp\ipykernel_10524\3578440984.py:48: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout(rect=[0, 0, 1, 0.96])


Figura salva em C:\Users\Administrator\Documents\Biopark\Projeto-Integrador-3\PI3-Grupo-03\docs\entrega\figures\pipeline_entrega_exemplos.png


C:\Users\Administrator\AppData\Local\Temp\ipykernel_10524\3578440984.py:53: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Leitura para a pergunta do projeto

Com crops centrados no nódulo, **IoU alto sozinho não prova** que a rede olha o nódulo:
o mapa centrado (sem aprendizado) passa de IoU 0,40 e tem Pointing Game ~100%.

No teste do v2 (provisório):

- **Grad-CAM** localiza mal em média (IoU ~0,11, Pointing Game ~42%) e frequentemente destaca pleura/parede.
- **IG** fica um pouco melhor que Grad-CAM, ainda abaixo do centrado.
- Conclusão operacional: as regiões destacadas **nem sempre** correspondem ao nódulo anotado;
  a comparação com o baseline centrado e os testes de deslocamento/Adebayo (Sprint 6)
  são necessários para interpretar a pergunta.

Documentação e figuras: `docs/entrega/`.